# 18 — Opt-in restaurant visit suggestions

**Fictional, offline tutorial.** The real feature takes only deliberately selected metadata, never photos or a camera roll. All proposals are unconfirmed and require human decisions. This notebook simulates decisions only to demonstrate the contract; it does not create real personal history.

In [ ]:
from pathlib import Path
import json
from savoroute.ingestion.osm import import_osm_snapshot
from savoroute.visits.workflow import suggest_visits, build_review_template, finalize_visits
base = Path("data/sample")
osm = json.loads((base / "osm_overpass_fixture_versioned.json").read_text())
metadata = json.loads((base / "visit_photo_metadata_fixture.json").read_text())
catalog = import_osm_snapshot(osm, synthetic_fixture=True).catalog
print("Synthetic catalog restaurants:", len(catalog.items), "Selected metadata records:", len(metadata["photos"]))

## Deterministic temporal/spatial clustering

All timestamps are normalized to UTC, and match candidates are located using OSM source identity and approximate geometry. Distances are not probabilities. There is no image inference.

In [ ]:
suggestions = suggest_visits(catalog, osm, metadata)
assert suggestions["status"] == "unreviewed"
assert [x["photo_count"] for x in suggestions["clusters"]] == [2, 2, 1]
for group in suggestions["clusters"]:
    print(group["visit_id"], group["photo_count"], [(c["title"], c["distance_m"]) for c in group["candidates"]])

## Explicit review gate

The generated review sheet starts with all decisions blank. In a real workflow, the user edits it privately in the `/visits-review` browser page (no upload) or in a local editor.

In [ ]:
review = build_review_template(suggestions)
assert all(row["decision"] is None for row in review["decisions"])
print("Unreviewed decisions:", len(review["decisions"]))

## **Synthetic decisions for tutorial only**

The following code simulates one confirmation, one explicit reassignment, and one rejection. Do not use these as judgments on real photo metadata.

In [ ]:
review["decisions"][0].update(decision="confirm", item_id=suggestions["clusters"][0]["candidates"][0]["item_id"], reason="Fictional user confirmation", reviewed_on="2026-01-06")
review["decisions"][1].update(decision="reassign", item_id=suggestions["clusters"][0]["candidates"][0]["item_id"], reason="Fictional user correction", reviewed_on="2026-01-06")
review["decisions"][2].update(decision="reject", item_id=None, reason="Fictional user rejection", reviewed_on="2026-01-06")
history = finalize_visits(catalog, suggestions, review)
assert history["rejected_count"] == 1
assert len(history["confirmed_visits"]) == 2
assert "latitude" not in json.dumps(history) and "demo_a" not in json.dumps(history)
print("Simulated confirmed:", len(history["confirmed_visits"]), "rejected:", history["rejected_count"])

## Limitations and next steps

This pipeline does not automatically infer or certify actual dining. It does not read images, run food detectors, upload user metadata, or change the canonical Savoroute catalog. For actual use, select and export only authorized metadata, review each suggestion, and avoid publishing location histories. See `docs/opt-in-visit-discovery.md`.